# SciGeoGuard-X — Final Publication Closure v9

## One notebook to close the remaining work before the FCS *Code & Data in Earth Science* submission

This notebook is **not another exploratory experiment**.

It performs the final closure tasks:

1. ingest the canonical SciGeoGuard-X GitHub/result package;
2. verify the v3 controlled benchmark, v6 strong 7B baselines, and v8/v8.1 real-data evidence;
3. detect whether v7 human review is actually complete;
4. detect whether a live v8.2 IMERG recovery result already exists;
5. if needed, run the resilient IMERG recovery experiment directly;
6. build the final evidence table;
7. rebuild the final publication figure;
8. audit the GitHub release for placeholders, missing license, oversized files, and likely secrets;
9. generate a final FCS claim set that never overstates pending evidence;
10. export a single publication/supplementary ZIP with hashes.

## Important scientific rules

- Generated benchmark labels are **controlled reference labels**, not independent human ground truth.
- LLM consensus is **not** human ground truth.
- v7 is considered complete only when actual reviewer verdicts exist.
- v8.2 offline fixtures are never accepted as scientific results.
- If v8.2 live IMERG recovery fails, the notebook automatically omits the multi-region IMERG empirical claim instead of blocking the rest of the paper.

## 0. Runtime configuration

Use a normal Colab CPU runtime. No GPU is required unless you intentionally rerun the earlier LLM notebooks.

You may provide either:

- the GitHub-ready repository ZIP produced from the previous work, or
- individual v3/v6/v8.1/v7/v8.2 result ZIPs/files.

The notebook recursively discovers canonical filenames.

In [ ]:
LIVE_IMERG_IF_NEEDED = True
SEED = 20261004

# Publication policy:
# Human review is strongly recommended but is NOT a hard gate for the short
# Code & Data software note because the manuscript already limits the benchmark
# claim to controlled consistency testing.
REQUIRE_V7_HUMAN_FOR_FCS = False

# A software license must be chosen before claiming a public open-source release.
# Leave UNSET until you deliberately choose one.
LICENSE_CHOICE = "UNSET"

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import os, re, io, json, math, time, random, shutil, zipfile, hashlib, tempfile
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v9"
INPUTS = WORK / "inputs"
EXTRACTED = WORK / "extracted"
OUTPUTS = WORK / "outputs"
CACHE = WORK / "http_cache"

for d in [WORK, INPUTS, EXTRACTED, OUTPUTS, CACHE]:
    d.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v9 publication closure")
print("ROOT:", ROOT)
print("WORK:", WORK)

# Part I — Upload or discover the current repository/results

In Colab, upload the GitHub repository ZIP and any later completed files, especially:

- `Reviewer_A_completed.csv`
- `Reviewer_B_completed.csv`
- `Adjudicator_completed.csv`
- a live `SciGeoGuard_X_v8_2_IMERG_Recovery.zip` if you already ran v8.2

If the files are already present in `/content` or `/mnt/data`, the notebook uses them automatically.

In [ ]:
def maybe_upload():
    if not Path("/content").exists():
        return []

    try:
        from google.colab import files
        uploaded = files.upload()
        paths=[]
        for name,data in uploaded.items():
            p=INPUTS/name
            p.write_bytes(data)
            paths.append(p)
        return paths
    except Exception as e:
        print("Upload UI skipped:", repr(e))
        return []

# Set to True only when running interactively in Colab and you need the upload dialog.
OPEN_UPLOAD_DIALOG = False

if OPEN_UPLOAD_DIALOG:
    uploaded_paths = maybe_upload()
    print("Uploaded:", [p.name for p in uploaded_paths])
else:
    print("OPEN_UPLOAD_DIALOG=False; scanning existing runtime files.")

In [ ]:
def discover_candidate_files():
    roots=[ROOT, INPUTS]
    seen={}
    for base in roots:
        if not base.exists():
            continue
        for p in base.rglob("*"):
            if p.is_file():
                try:
                    key=str(p.resolve())
                except Exception:
                    key=str(p)
                seen[key]=p
    return list(seen.values())

candidate_files=discover_candidate_files()
print("Runtime files discovered:", len(candidate_files))

# Display likely SciGeoGuard inputs.
interesting=[
    p for p in candidate_files
    if any(k in p.name.lower() for k in [
        "scigeoguard","reviewer_a","reviewer_b","adjudicator",
        "strong_llm","silentgeobench","worldcover","copdem","imerg","sentinel"
    ])
]
for p in sorted(interesting)[:200]:
    print(p)

# Part II — Extract ZIP inputs into one searchable workspace

ZIPs are expanded into separate directories. Original archives remain unchanged.

In [ ]:
def safe_extract_zip(zpath, dest):
    zpath=Path(zpath)
    dest=Path(dest)
    dest.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            if info.is_dir():
                continue

            rel=Path(info.filename)

            # Reject absolute/path-traversal entries.
            if rel.is_absolute() or ".." in rel.parts:
                raise RuntimeError(
                    f"Unsafe ZIP entry in {zpath.name}: {info.filename}"
                )

            out=dest/rel
            out.parent.mkdir(parents=True, exist_ok=True)

            with z.open(info) as fi, open(out,"wb") as fo:
                shutil.copyfileobj(fi,fo)

zip_inputs=[
    p for p in candidate_files
    if p.suffix.lower()==".zip" and
    (
        "scigeoguard" in p.name.lower() or
        "review" in p.name.lower()
    )
]

for i,zp in enumerate(sorted(set(zip_inputs))):
    target=EXTRACTED/f"{i:03d}_{zp.stem}"
    try:
        safe_extract_zip(zp,target)
        print("Extracted:",zp.name,"->",target)
    except Exception as e:
        print("ZIP extraction failed:",zp,repr(e))

In [ ]:
SEARCH_ROOTS=[ROOT, INPUTS, EXTRACTED]

def all_files():
    out=[]
    seen=set()

    for r in SEARCH_ROOTS:
        if not r.exists():
            continue

        for p in r.rglob("*"):
            if not p.is_file():
                continue

            try:
                k=str(p.resolve())
            except Exception:
                k=str(p)

            if k not in seen:
                seen.add(k)
                out.append(p)

    return out

ALL_FILES=all_files()

def find_named(filename):
    hits=[p for p in ALL_FILES if p.name==filename]
    # Prefer non-dry-run, extracted actual result files, then newest/largest.
    hits.sort(
        key=lambda p:(
            "dryrun" in str(p).lower() or "offline" in str(p).lower(),
            -p.stat().st_size
        )
    )
    return hits

def first_named(filename):
    hits=find_named(filename)
    return hits[0] if hits else None

print("Searchable files:",len(ALL_FILES))

# Part III — Canonical evidence discovery

This section discovers the exact result files produced by the earlier notebooks.

In [ ]:
CANONICAL_NAMES = {
    "v3_benchmark":[
        "SilentGeoBench_v3.csv"
    ],
    "v3_metrics":[
        "SciGeoGuard_X_v3_metrics.csv"
    ],
    "v6_metrics":[
        "Strong_LLM_verdict_metrics.csv"
    ],
    "v6_health":[
        "Strong_LLM_health_summary.csv"
    ],
    "v8_sentinel":[
        "Sentinel2_multiregion_results.csv",
        "Sentinel2_v8_preserved_live_results.csv"
    ],
    "v8_worldcover":[
        "WorldCover_multiregion_results.csv",
        "WorldCover_v8_preserved_live_results.csv"
    ],
    "v81_copdem":[
        "CopDEM_v8_1_corrected_results.csv"
    ],
    "v81_pub":[
        "FINAL_PUBLICATION_RESULTS_TABLE_v8_1.csv"
    ],
    "v81_boot":[
        "Region_cluster_bootstrap_95CI.csv"
    ],
    "v81_gates":[
        "PUBLICATION_GATES_v8_1.csv"
    ],
    "v82_imerg":[
        "IMERG_v8_2_recovered_regions.csv"
    ],
    "v82_gates":[
        "PUBLICATION_GATES_IMERG_v8_2.csv"
    ],
    "v7_A":[
        "Reviewer_A_completed.csv"
    ],
    "v7_B":[
        "Reviewer_B_completed.csv"
    ],
    "v7_C":[
        "Adjudicator_completed.csv"
    ],
    "v7_internal":[
        "INTERNAL_60_case_mapping_and_machine_context.csv"
    ]
}

FOUND={}

for key,names in CANONICAL_NAMES.items():
    p=None
    for name in names:
        q=first_named(name)
        if q is not None:
            p=q
            break

    FOUND[key]=p
    print(f"{key:12s}:", p)

# Part IV — Read and validate the completed evidence

The notebook refuses to treat a dry-run fixture as a live scientific result.

In [ ]:
def read_csv_if(path):
    if path is None:
        return pd.DataFrame()
    return pd.read_csv(path)

data={k:read_csv_if(v) for k,v in FOUND.items()}

def contains_fixture_marker(df):
    if df.empty:
        return False

    for col in df.columns:
        if df[col].dtype==object:
            vals=" ".join(
                df[col].astype(str)
                .head(10000)
                .str.lower()
                .tolist()
            )
            if "offline_fixture" in vals or "synthetic_fixture" in vals:
                return True

    return False

fixture_flags={
    k:contains_fixture_marker(df)
    for k,df in data.items()
}

fixture_flags

In [ ]:
evidence_checks=[]

# v3
v3=data["v3_benchmark"]
if not v3.empty:
    expected_col = (
        "expected"
        if "expected" in v3.columns
        else "generated_expected"
        if "generated_expected" in v3.columns
        else None
    )

    v3_counts=(
        v3[expected_col].value_counts().to_dict()
        if expected_col else {}
    )

    evidence_checks.append({
        "component":"v3 controlled benchmark",
        "available":True,
        "valid_for_claims":not fixture_flags["v3_benchmark"],
        "detail":
            f"{len(v3)} cases; labels={v3_counts}"
    })
else:
    v3_counts={}
    evidence_checks.append({
        "component":"v3 controlled benchmark",
        "available":False,
        "valid_for_claims":False,
        "detail":"SilentGeoBench_v3.csv not found"
    })

# v6
v6=data["v6_metrics"]
if not v6.empty:
    evidence_checks.append({
        "component":"v6 strong 7B baselines",
        "available":True,
        "valid_for_claims":not fixture_flags["v6_metrics"],
        "detail":
            "; ".join(
                f"{r.get('system','?')}: acc={pd.to_numeric(r.get('accuracy_vs_generated_reference',np.nan),errors='coerce'):.4f}, "
                f"FSCR={pd.to_numeric(r.get('FSCR_vs_generated_reference',np.nan),errors='coerce'):.4f}"
                for _,r in v6.iterrows()
            )
    })
else:
    evidence_checks.append({
        "component":"v6 strong 7B baselines",
        "available":False,
        "valid_for_claims":False,
        "detail":"Strong_LLM_verdict_metrics.csv not found"
    })

# v8.1
g81=data["v81_gates"]
if not g81.empty and "passed" in g81.columns:
    pcount=int(
        g81["passed"]
        .astype(str)
        .str.lower()
        .map({"true":True,"false":False})
        .fillna(g81["passed"])
        .astype(bool)
        .sum()
    )
    evidence_checks.append({
        "component":"v8.1 corrected live EO",
        "available":True,
        "valid_for_claims":not fixture_flags["v81_gates"],
        "detail":f"{pcount}/{len(g81)} v8.1 gates passed"
    })
else:
    evidence_checks.append({
        "component":"v8.1 corrected live EO",
        "available":False,
        "valid_for_claims":False,
        "detail":"v8.1 publication gates not found"
    })

evidence_check_df=pd.DataFrame(evidence_checks)
display(evidence_check_df)

# Part V — Determine the remaining work

There are only three possible open scientific/release tasks:

1. **v8.2 IMERG live recovery** — optional for the final short note; required only if the manuscript wants to claim multi-region rainy IMERG replication.
2. **v7 independent human validation** — strongly useful, but not a hard gate for the short Code & Data note if all benchmark claims remain explicitly controlled/reference-based.
3. **public release metadata** — mandatory before calling the resource public/open-source: license, repository URL, corresponding email, and preferably an archival DOI.

In [ ]:
# v7 completion
A=data["v7_A"]
B=data["v7_B"]
C=data["v7_C"]

def reviewer_completed(df):
    if df.empty or "reviewer_verdict" not in df.columns:
        return False
    vals=df["reviewer_verdict"].astype(str).str.strip().str.upper()
    return len(df)>=1 and vals.isin({"PROVEN","REFUTED","UNPROVEN"}).all()

V7_A_DONE=reviewer_completed(A)
V7_B_DONE=reviewer_completed(B)
V7_DOUBLE_REVIEW_DONE=V7_A_DONE and V7_B_DONE

# v8.2 completion
v82=data["v82_imerg"]
g82=data["v82_gates"]

def bool_series(s):
    if s.dtype==bool:
        return s
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map({"true":True,"false":False})
    )

V82_LIVE_READY=False

if not v82.empty and not fixture_flags["v82_imerg"]:
    live_ok=True

    if "status" in v82:
        live_ok &= bool((v82["status"].astype(str)=="SUCCESS").all())

    if "region" in v82:
        live_ok &= int(v82["region"].nunique())>=4

    if not g82.empty and "passed" in g82.columns:
        bs=bool_series(g82["passed"])
        live_ok &= bool(bs.fillna(False).all())

    V82_LIVE_READY=bool(live_ok)

remaining_df=pd.DataFrame([
    {
        "task":"v8.2 multi-region rainy IMERG recovery",
        "status":"DONE" if V82_LIVE_READY else "OPEN",
        "hard_gate_for_FCS":False,
        "action":
            "Use result in paper"
            if V82_LIVE_READY
            else "Run recovery below or omit multi-region IMERG claim"
    },
    {
        "task":"v7 independent double human review",
        "status":"DONE" if V7_DOUBLE_REVIEW_DONE else "OPEN",
        "hard_gate_for_FCS":REQUIRE_V7_HUMAN_FOR_FCS,
        "action":
            "Compute final human comparison"
            if V7_DOUBLE_REVIEW_DONE
            else "Recommended follow-up; do not claim human-validated accuracy"
    },
    {
        "task":"public software license",
        "status":"DONE" if LICENSE_CHOICE!="UNSET" else "OPEN",
        "hard_gate_for_FCS":True,
        "action":
            "License selected"
            if LICENSE_CHOICE!="UNSET"
            else "Choose license deliberately before public release"
    }
])

display(remaining_df)

# Part VI — Resilient live IMERG recovery (runs only if needed)

This is the v8.2 recovery logic integrated into the final closure notebook.

It searches wet-season candidate dates for one rainy, 48-half-hour day in each of:

- Bengaluru
- Rotterdam
- California Central Valley
- Manaus

A tiny one-cell scout is used first; only promising dates get a 3×3 neighborhood query.

In [ ]:
RUN_IMERG_RECOVERY = (
    LIVE_IMERG_IF_NEEDED and
    not V82_LIVE_READY
)

REGIONS = {
    "Bengaluru_India": {
        "center": (12.98, 77.59),
        "windows": [("2024-06-01","2024-10-31")]
    },
    "Rotterdam_Netherlands": {
        "center": (51.94, 4.47),
        "windows": [
            ("2024-09-01","2024-12-15"),
            ("2024-01-01","2024-03-31")
        ]
    },
    "CentralValley_USA": {
        "center": (38.49, -121.81),
        "windows": [
            ("2024-01-01","2024-03-31"),
            ("2024-11-01","2024-12-31")
        ]
    },
    "Manaus_Brazil": {
        "center": (-3.12, -60.04),
        "windows": [("2024-01-01","2024-05-31")]
    }
}

BASE = (
    "https://erddap.aoml.noaa.gov/hdb/erddap/griddap/"
    "precipitation_2024_v07"
)

INTERVAL_H=0.5
EXPECTED_INTERVALS=48
LAT_START=-89.95
LON_START=-179.95
STEP=0.1
RAIN_THRESHOLD_MM=1.0
MAX_CANDIDATES=30

print("RUN_IMERG_RECOVERY =",RUN_IMERG_RECOVERY)

In [ ]:
if RUN_IMERG_RECOVERY:
    import sys, subprocess
    subprocess.check_call([
        sys.executable,"-m","pip","install","-q",
        "requests>=2.31","urllib3>=2.0"
    ])

    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    retry=Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2.0,
        status_forcelist=[429,500,502,503,504],
        allowed_methods=["GET"],
        raise_on_status=False
    )

    SESSION=requests.Session()
    SESSION.headers.update({
        "User-Agent":"SciGeoGuard-X-v9/1.0"
    })
    SESSION.mount(
        "https://",
        HTTPAdapter(max_retries=retry)
    )
else:
    SESSION=None

def cached_get(url, timeout=(30,180)):
    h=hashlib.sha256(url.encode()).hexdigest()
    p=CACHE/(h+".bin")

    if p.exists():
        return p.read_bytes()

    r=SESSION.get(url,timeout=timeout)

    if r.status_code>=400:
        raise RuntimeError(
            f"HTTP {r.status_code}"
        )

    p.write_bytes(r.content)
    return r.content

def cached_csv(url):
    b=cached_get(url)

    if not b.strip():
        return pd.DataFrame()

    try:
        return pd.read_csv(io.BytesIO(b))
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def detect_col(df,needle):
    hits=[
        c for c in df.columns
        if needle.lower() in c.lower()
    ]

    if not hits:
        raise KeyError(
            f"No {needle!r} column in {list(df.columns)}"
        )

    return hits[0]

def nearest_center(value,start):
    return round(
        start +
        round((value-start)/STEP)*STEP,
        2
    )

def time_url(date):
    return (
        BASE +
        ".csv?time"
        f"[({date}T00:00:00Z):1:({date}T23:30:00Z)]"
    )

def precip_url(date,lat0,lat1,lon0,lon1):
    return (
        BASE +
        ".csv?precipitation"
        f"[({date}T00:00:00Z):1:({date}T23:30:00Z)]"
        f"[({lat0:.2f}):1:({lat1:.2f})]"
        f"[({lon0:.2f}):1:({lon1:.2f})]"
    )

def candidate_dates(windows,step_days=4):
    vals=[]

    for start,end in windows:
        d=pd.Timestamp(start)
        e=pd.Timestamp(end)

        while d<=e:
            vals.append(d.strftime("%Y-%m-%d"))
            d+=pd.Timedelta(days=step_days)

    rng=np.random.default_rng(
        SEED+len(vals)
    )

    order=rng.permutation(len(vals))

    return [
        vals[i]
        for i in order[:MAX_CANDIDATES]
    ]

def validate_time_axis(date):
    d=cached_csv(time_url(date))

    if d.empty:
        return False,0

    tcol=detect_col(d,"time")

    times=(
        pd.to_datetime(
            d[tcol],
            errors="coerce",
            utc=True
        )
        .dropna()
        .drop_duplicates()
    )

    return (
        len(times)==EXPECTED_INTERVALS,
        int(len(times))
    )

def scout_day(date,lat,lon):
    latc=nearest_center(lat,LAT_START)
    lonc=nearest_center(lon,LON_START)

    d=cached_csv(
        precip_url(
            date,
            latc,latc,
            lonc,lonc
        )
    )

    if d.empty:
        rate_sum=0.0
        n=0
    else:
        pcol=detect_col(d,"precip")

        p=(
            pd.to_numeric(
                d[pcol],
                errors="coerce"
            )
            .dropna()
        )

        rate_sum=float(p.sum())
        n=int(len(p))

    return {
        "lat":latc,
        "lon":lonc,
        "rate_sum":rate_sum,
        "accum_mm":rate_sum*INTERVAL_H,
        "returned_records":n
    }

def confirm_3x3(region,date,lat,lon):
    latc=nearest_center(lat,LAT_START)
    lonc=nearest_center(lon,LON_START)

    lats=np.array([
        latc-0.1,
        latc,
        latc+0.1
    ])

    lons=np.array([
        lonc-0.1,
        lonc,
        lonc+0.1
    ])

    d=cached_csv(
        precip_url(
            date,
            lats.min(),lats.max(),
            lons.min(),lons.max()
        )
    )

    expected_cells=(
        pd.MultiIndex
        .from_product(
            [
                np.round(lats,2),
                np.round(lons,2)
            ],
            names=[
                "latitude",
                "longitude"
            ]
        )
        .to_frame(index=False)
    )

    if d.empty:
        grouped=pd.DataFrame(
            columns=[
                "latitude",
                "longitude",
                "rate_sum",
                "returned_records"
            ]
        )
    else:
        pcol=detect_col(d,"precip")
        latcol=detect_col(d,"lat")
        loncol=detect_col(d,"lon")

        d[pcol]=pd.to_numeric(
            d[pcol],
            errors="coerce"
        )

        d[latcol]=pd.to_numeric(
            d[latcol],
            errors="coerce"
        )

        d[loncol]=pd.to_numeric(
            d[loncol],
            errors="coerce"
        )

        d=d.dropna(
            subset=[
                pcol,
                latcol,
                loncol
            ]
        ).copy()

        d["latitude"]=d[latcol].round(2)
        d["longitude"]=d[loncol].round(2)

        grouped=(
            d.groupby([
                "latitude",
                "longitude"
            ])[pcol]
            .agg(
                rate_sum="sum",
                returned_records="count"
            )
            .reset_index()
        )

    cells=expected_cells.merge(
        grouped,
        on=[
            "latitude",
            "longitude"
        ],
        how="left"
    )

    cells["rate_sum"]=(
        pd.to_numeric(
            cells["rate_sum"],
            errors="coerce"
        )
        .fillna(0.0)
    )

    cells["returned_records"]=(
        pd.to_numeric(
            cells["returned_records"],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )

    cells["correct_accum_mm"]=(
        cells.rate_sum*INTERVAL_H
    )

    cells["raw_sum_numeric"]=cells.rate_sum

    cells["full_day_mean_numeric"]=(
        cells.rate_sum /
        EXPECTED_INTERVALS
    )

    rainy=cells[
        cells.correct_accum_mm >=
        RAIN_THRESHOLD_MM
    ].copy()

    if len(rainy)==0:
        raise RuntimeError(
            "No >=1 mm rainy cells in 3x3 confirmation."
        )

    rainy["raw_factor"]=(
        rainy.raw_sum_numeric /
        rainy.correct_accum_mm
    )

    rainy["mean_fraction"]=(
        rainy.full_day_mean_numeric /
        rainy.correct_accum_mm
    )

    return {
        "region":region,
        "selected_date":date,
        "status":"SUCCESS",
        "grid_cells":int(len(cells)),
        "rainy_cells_ge_1mm":
            int(len(rainy)),
        "mean_daily_accum_mm":
            float(cells.correct_accum_mm.mean()),
        "max_daily_accum_mm":
            float(cells.correct_accum_mm.max()),
        "median_rainy_accum_mm":
            float(rainy.correct_accum_mm.median()),
        "median_raw_sum_factor":
            float(rainy.raw_factor.median()),
        "median_mean_as_accum_fraction":
            float(rainy.mean_fraction.median()),
        "returned_nonzero_records":
            int(cells.returned_records.sum()),
        "expected_records":
            int(len(cells)*EXPECTED_INTERVALS),
        "reconstructed_zero_fraction":
            float(
                1 -
                cells.returned_records.sum()/
                (len(cells)*EXPECTED_INTERVALS)
            )
    }

def recover_region(region,cfg):
    lat,lon=cfg["center"]
    log=[]

    for date in candidate_dates(
        cfg["windows"]
    ):
        rec={
            "region":region,
            "date":date
        }

        try:
            scout=scout_day(
                date,
                lat,
                lon
            )

            rec.update(scout)

            if scout["accum_mm"] < RAIN_THRESHOLD_MM:
                rec["decision"]="DRY_SCOUT"
                log.append(rec)
                continue

            ok,n=validate_time_axis(date)
            rec["time_axis_count"]=n

            if not ok:
                rec["decision"]="TIME_AXIS_FAIL"
                log.append(rec)
                continue

            final=confirm_3x3(
                region,
                date,
                lat,
                lon
            )

            rec["decision"]="SELECTED"
            log.append(rec)

            return final,log

        except Exception as e:
            rec["decision"]="REQUEST_ERROR"
            rec["error"]=repr(e)
            log.append(rec)

    raise RuntimeError(
        f"No verified rainy day found for {region}"
    )

In [ ]:
imerg_log=[]
imerg_recovered=[]

if RUN_IMERG_RECOVERY:
    for region,cfg in REGIONS.items():
        try:
            result,log=recover_region(
                region,
                cfg
            )

            imerg_recovered.append(result)
            imerg_log.extend(log)

            print(
                "Recovered:",
                region,
                result["selected_date"],
                "median rainy accumulation",
                result["median_rainy_accum_mm"]
            )

        except Exception as e:
            imerg_recovered.append({
                "region":region,
                "status":"FAILED",
                "error":repr(e)
            })

            print(
                "FAILED:",
                region,
                repr(e)
            )

    v82_final=pd.DataFrame(
        imerg_recovered
    )

    imerg_log_df=pd.DataFrame(
        imerg_log
    )

else:
    v82_final=v82.copy()
    imerg_log_df=pd.DataFrame()

display(v82_final)

In [ ]:
if not v82_final.empty:
    success=v82_final[
        v82_final.get(
            "status",
            pd.Series("",index=v82_final.index)
        ).astype(str).eq("SUCCESS")
    ].copy()
else:
    success=pd.DataFrame()

V82_FINAL_READY=(
    len(success)==4 and
    success["region"].nunique()==4 and
    "median_raw_sum_factor" in success and
    "median_mean_as_accum_fraction" in success and
    np.allclose(
        pd.to_numeric(
            success.median_raw_sum_factor,
            errors="coerce"
        ),
        2.0,
        rtol=1e-6,
        atol=1e-8
    ) and
    np.allclose(
        pd.to_numeric(
            success.median_mean_as_accum_fraction,
            errors="coerce"
        ),
        1/24,
        rtol=1e-6,
        atol=1e-8
    )
)

print(
    "V82_FINAL_READY =",
    V82_FINAL_READY
)

# Part VII — Optional v7 human-review synthesis

If completed Reviewer A/B files are present, compute inter-rater agreement.

If adjudication is incomplete, the notebook does **not** invent a final human label.

In [ ]:
VALID_LABELS={
    "PROVEN",
    "REFUTED",
    "UNPROVEN"
}

def cohens_kappa(a,b):
    a=pd.Series(a)
    b=pd.Series(b)

    ok=(
        a.isin(VALID_LABELS) &
        b.isin(VALID_LABELS)
    )

    a=a[ok]
    b=b[ok]

    if len(a)==0:
        return np.nan

    po=float((a.values==b.values).mean())

    pa=a.value_counts(
        normalize=True
    )

    pb=b.value_counts(
        normalize=True
    )

    pe=sum(
        float(pa.get(x,0))*
        float(pb.get(x,0))
        for x in VALID_LABELS
    )

    return (
        np.nan
        if math.isclose(1-pe,0)
        else (po-pe)/(1-pe)
    )

human_summary=pd.DataFrame()

if V7_DOUBLE_REVIEW_DONE:
    A2=A.copy()
    B2=B.copy()

    A2["reviewer_verdict"]=(
        A2["reviewer_verdict"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    B2["reviewer_verdict"]=(
        B2["reviewer_verdict"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    pair=(
        A2[[
            "human_case_id",
            "reviewer_verdict"
        ]]
        .rename(columns={
            "reviewer_verdict":
                "reviewer_A"
        })
        .merge(
            B2[[
                "human_case_id",
                "reviewer_verdict"
            ]].rename(columns={
                "reviewer_verdict":
                    "reviewer_B"
            }),
            on="human_case_id",
            how="inner"
        )
    )

    human_summary=pd.DataFrame([{
        "n_double_reviewed":len(pair),
        "raw_agreement":
            float(
                (
                    pair.reviewer_A ==
                    pair.reviewer_B
                ).mean()
            ),
        "cohens_kappa":
            cohens_kappa(
                pair.reviewer_A,
                pair.reviewer_B
            ),
        "disagreements":
            int(
                (
                    pair.reviewer_A !=
                    pair.reviewer_B
                ).sum()
            )
    }])

    display(human_summary)

else:
    print(
        "No completed double-human review found. "
        "v7 remains pending and no human-validated accuracy will be claimed."
    )

# Part VIII — Build the final evidence table

Only completed evidence enters the main publication table.

In [ ]:
rows=[]

# v3
if not v3.empty:
    rows.append({
        "evidence_layer":
            "Controlled scientific-invariant benchmark",
        "version":"v3",
        "status":"COMPLETE",
        "result":
            f"{len(v3)} controlled cases; "
            f"{v3_counts.get('PROVEN',0)} PROVEN, "
            f"{v3_counts.get('REFUTED',0)} REFUTED, "
            f"{v3_counts.get('UNPROVEN',0)} UNPROVEN",
        "claim_boundary":
            "Controlled reference/consistency test; not independent human ground truth"
    })

# v6
if not v6.empty:
    for _,r in v6.iterrows():
        rows.append({
            "evidence_layer":
                "Strong decomposed LLM baseline",
            "version":"v6",
            "status":"COMPLETE",
            "result":
                f"{r.get('system','LLM')}: "
                f"accuracy={pd.to_numeric(r.get('accuracy_vs_generated_reference',np.nan),errors='coerce'):.4f}; "
                f"FSCR={pd.to_numeric(r.get('FSCR_vs_generated_reference',np.nan),errors='coerce'):.4f}",
            "claim_boundary":
                "Baseline against generated reference; not a ground-truth judge"
        })

# v8.1 summary
pub81=data["v81_pub"]

if not pub81.empty:
    for _,r in pub81.iterrows():
        exp=str(r.get("experiment",""))

        # Skip IMERG here; v8.2 decides whether it enters final empirical table.
        if "IMERG" in exp.upper():
            continue

        rows.append({
            "evidence_layer":
                "Live Earth-observation counterfactual",
            "version":"v8.1",
            "status":"COMPLETE",
            "result":
                f"{exp}: primary={r.get('primary_effect',np.nan)}; "
                f"secondary={r.get('secondary_effect',np.nan)}",
            "claim_boundary":
                "Real-input counterfactual scientific stress test; not human annotation"
        })

# v8.2 IMERG
if V82_FINAL_READY:
    rows.append({
        "evidence_layer":
            "Live Earth-observation counterfactual",
        "version":"v8.2",
        "status":"COMPLETE",
        "result":
            "IMERG: 4/4 regions recovered with rainy days; "
            "raw rate-sum factor=2.0; mean-rate/accumulation=1/24",
        "claim_boundary":
            "Dimensional-quantity stress test on live IMERG; deterministic identity confirmed across selected rainy regions"
    })
else:
    rows.append({
        "evidence_layer":
            "Live IMERG recovery",
        "version":"v8.2",
        "status":"NOT INCLUDED IN MAIN CLAIMS",
        "result":
            "Multi-region rainy recovery not closed",
        "claim_boundary":
            "Omit multi-region IMERG empirical claim from manuscript"
    })

# v7
rows.append({
    "evidence_layer":
        "Independent blinded human validation",
    "version":"v7",
    "status":
        "COMPLETE"
        if V7_DOUBLE_REVIEW_DONE
        else "PENDING",
    "result":
        (
            f"Double review complete; "
            f"agreement={human_summary.iloc[0].raw_agreement:.4f}, "
            f"kappa={human_summary.iloc[0].cohens_kappa:.4f}"
        )
        if not human_summary.empty
        else
        "Reviewer sheets exist but completed human verdicts were not found",
    "claim_boundary":
        (
            "Human agreement available; adjudication may still be needed"
            if V7_DOUBLE_REVIEW_DONE
            else
            "Do not claim independently human-validated system accuracy"
        )
})

final_evidence_df=pd.DataFrame(rows)
display(final_evidence_df)

# Part IX — Rebuild the final publication figure

The figure uses only evidence that is available and scientifically cleared.

Panels:

- A: evidence stack/status;
- B: Sentinel-2 spectral-role effect by region;
- C: WorldCover invalid categorical-code fraction;
- D: CopDEM corrected p95 round-trip error;
- optional IMERG annotation when v8.2 is complete.

This is generated as separate matplotlib figures and then stitched into one composite image.

In [ ]:
sentinel=data["v8_sentinel"]
worldcover=data["v8_worldcover"]
copdem=data["v81_copdem"]

# Normalize to successful live rows.
if not sentinel.empty and "status" in sentinel:
    sentinel=sentinel[
        sentinel.status.astype(str).eq("SUCCESS")
    ].copy()

if not worldcover.empty and "status" in worldcover:
    worldcover=worldcover[
        worldcover.status.astype(str).eq("SUCCESS")
    ].copy()

if not copdem.empty and "status" in copdem:
    copdem=copdem[
        copdem.status.astype(str).eq("SUCCESS")
    ].copy()

# Region aggregation.
if not sentinel.empty:
    sentinel_region=(
        sentinel.groupby("region")
        .agg(
            spectral_mae=(
                "mean_absolute_index_difference",
                "median"
            )
        )
        .reset_index()
    )
else:
    sentinel_region=pd.DataFrame()

if not worldcover.empty:
    world_region=(
        worldcover.groupby("region")
        .agg(
            invalid_fraction=(
                "bilinear_invalid_fraction",
                "median"
            )
        )
        .reset_index()
    )
else:
    world_region=pd.DataFrame()

if not copdem.empty:
    copdem_region=(
        copdem.groupby("region")
        .agg(
            p95_error_m=(
                "roundtrip_core_p95_abs_error_m",
                "median"
            ),
            derived_fraction=(
                "derived_output_fraction_without_distinct_source_sample",
                "median"
            )
        )
        .reset_index()
    )
else:
    copdem_region=pd.DataFrame()

print("Sentinel regions:",len(sentinel_region))
print("WorldCover regions:",len(world_region))
print("CopDEM regions:",len(copdem_region))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

FIGDIR=OUTPUTS/"figure_panels"
FIGDIR.mkdir(exist_ok=True)

pretty={
    "Bengaluru_India":"Bengaluru",
    "Rotterdam_Netherlands":"Rotterdam",
    "CentralValley_USA":"Central Valley",
    "Manaus_Brazil":"Manaus"
}

panels=[]

# Panel A — evidence stack
p=FIGDIR/"A_evidence_stack.png"

fig,ax=plt.subplots(figsize=(7,4.5))
ax.axis("off")

summary_lines=[
    "Controlled benchmark — v3: " +
    ("available" if not v3.empty else "missing"),
    "Strong 7B baselines — v6: " +
    ("available" if not v6.empty else "missing"),
    "Live EO replication — v8.1: " +
    ("available" if not pub81.empty else "missing"),
    "Live IMERG recovery — v8.2: " +
    ("complete" if V82_FINAL_READY else "omitted/pending"),
    "Human review — v7: " +
    ("complete" if V7_DOUBLE_REVIEW_DONE else "pending")
]

ax.text(
    0.04,
    0.95,
    "SciGeoGuard-X evidence stack",
    va="top",
    fontsize=13,
    fontweight="bold"
)

for i,line in enumerate(summary_lines):
    ax.text(
        0.06,
        0.80-i*0.14,
        line,
        fontsize=10
    )

fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# Panel B — Sentinel
p=FIGDIR/"B_sentinel.png"
fig,ax=plt.subplots(figsize=(7,4.5))

if not sentinel_region.empty:
    labels=[
        pretty.get(x,x)
        for x in sentinel_region.region
    ]

    ax.bar(
        labels,
        sentinel_region.spectral_mae
    )

    ax.set_ylabel(
        "Regional median mean absolute index difference"
    )

    ax.tick_params(
        axis="x",
        rotation=20
    )
else:
    ax.text(.5,.5,"Sentinel result file missing",ha="center")

ax.set_title("Sentinel-2 spectral-role mismatch")
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# Panel C — WorldCover
p=FIGDIR/"C_worldcover.png"
fig,ax=plt.subplots(figsize=(7,4.5))

if not world_region.empty:
    labels=[
        pretty.get(x,x)
        for x in world_region.region
    ]

    ax.bar(
        labels,
        world_region.invalid_fraction
    )

    ax.set_ylabel(
        "Bilinear invalid class-code fraction"
    )

    ax.tick_params(
        axis="x",
        rotation=20
    )
else:
    ax.text(.5,.5,"WorldCover result file missing",ha="center")

ax.set_title("ESA WorldCover categorical interpolation")
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# Panel D — CopDEM
p=FIGDIR/"D_copdem.png"
fig,ax=plt.subplots(figsize=(7,4.5))

if not copdem_region.empty:
    labels=[
        pretty.get(x,x)
        for x in copdem_region.region
    ]

    ax.bar(
        labels,
        copdem_region.p95_error_m
    )

    ax.set_ylabel(
        "Aligned core p95 round-trip error (m)"
    )

    ax.tick_params(
        axis="x",
        rotation=20
    )
else:
    ax.text(.5,.5,"CopDEM result file missing",ha="center")

ax.set_title("Copernicus DEM aligned resampling")
fig.tight_layout()
fig.savefig(p,dpi=300,bbox_inches="tight")
plt.close(fig)
panels.append(p)

# Stitch 2x2.
imgs=[
    Image.open(x).convert("RGB")
    for x in panels
]

w=max(i.width for i in imgs)
h=max(i.height for i in imgs)

prepared=[]

for idx,im in enumerate(imgs):
    c=Image.new(
        "RGB",
        (w,h),
        "white"
    )

    c.paste(
        im,
        (
            (w-im.width)//2,
            (h-im.height)//2
        )
    )

    ImageDraw.Draw(c).text(
        (18,14),
        chr(ord("A")+idx),
        fill="black"
    )

    prepared.append(c)

composite=Image.new(
    "RGB",
    (2*w,2*h),
    "white"
)

composite.paste(prepared[0],(0,0))
composite.paste(prepared[1],(w,0))
composite.paste(prepared[2],(0,h))
composite.paste(prepared[3],(w,h))

FINAL_FIG=OUTPUTS/"FINAL_FCS_COMPOSITE_FIGURE_v9.png"
composite.save(FINAL_FIG,dpi=(300,300))

print(FINAL_FIG)

# Part X — GitHub/public-release audit

This checks the repository/package currently available in the runtime.

It does **not** upload anything to GitHub.

In [ ]:
# Prefer an extracted SciGeoGuard-X repository tree if present.
repo_candidates=[]

for p in [ROOT,EXTRACTED,INPUTS]:
    if not p.exists():
        continue

    for q in p.rglob("README.md"):
        parent=q.parent

        if (
            (parent/"notebooks").exists() and
            (parent/"results").exists()
        ):
            repo_candidates.append(parent)

repo_candidates=sorted(
    set(repo_candidates),
    key=lambda p:len(str(p))
)

REPO_ROOT=(
    repo_candidates[0]
    if repo_candidates
    else None
)

print("REPO_ROOT =",REPO_ROOT)

In [ ]:
SECRET_PATTERNS = {
    "OpenAI key":
        re.compile(r"sk-[A-Za-z0-9_-]{20,}"),

    "GitHub PAT":
        re.compile(r"gh[pousr]_[A-Za-z0-9]{20,}"),

    "AWS access key":
        re.compile(r"AKIA[0-9A-Z]{16}"),

    "Generic Hugging Face token":
        re.compile(r"hf_[A-Za-z0-9]{20,}")
}

TEXT_EXTS={
    ".py",".ipynb",".md",".txt",".json",
    ".yaml",".yml",".toml",".ini",".cfg",
    ".tex",".csv"
}

audit_rows=[]
secret_hits=[]

if REPO_ROOT is not None:
    repo_files=[
        p for p in REPO_ROOT.rglob("*")
        if p.is_file()
    ]

    oversized=[
        p for p in repo_files
        if p.stat().st_size > 100*1024*1024
    ]

    for p in repo_files:
        if p.suffix.lower() not in TEXT_EXTS:
            continue

        try:
            text=p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for name,pat in SECRET_PATTERNS.items():
            if pat.search(text):
                secret_hits.append({
                    "file":str(p.relative_to(REPO_ROOT)),
                    "pattern":name
                })

    placeholder_patterns=[
        "REPLACE-BEFORE-SUBMISSION",
        "REPLACE-WITH-CORRESPONDING-EMAIL",
        "[GITHUB-URL]",
        "[ZENODO-DOI]",
        "[corresponding-author-email]"
    ]

    placeholder_hits=[]

    for p in repo_files:
        if p.suffix.lower() not in TEXT_EXTS:
            continue

        try:
            text=p.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        except Exception:
            continue

        for ph in placeholder_patterns:
            if ph in text:
                placeholder_hits.append({
                    "file":
                        str(p.relative_to(REPO_ROOT)),
                    "placeholder":ph
                })

    has_license=any(
        p.name.lower().startswith("license")
        and
        p.name.lower() not in {
            "license_status.md"
        }
        for p in repo_files
    )

    audit_rows.extend([
        {
            "gate":"README present",
            "passed":
                (REPO_ROOT/"README.md").exists(),
            "detail":""
        },
        {
            "gate":"CITATION.cff present",
            "passed":
                (REPO_ROOT/"CITATION.cff").exists(),
            "detail":""
        },
        {
            "gate":"Explicit software LICENSE present",
            "passed":
                has_license,
            "detail":
                "Select a license deliberately; LICENSE_STATUS.md alone is not a license."
        },
        {
            "gate":"No files >100 MB",
            "passed":
                len(oversized)==0,
            "detail":
                "; ".join(
                    str(p.relative_to(REPO_ROOT))
                    for p in oversized[:10]
                )
        },
        {
            "gate":"No likely secrets",
            "passed":
                len(secret_hits)==0,
            "detail":
                json.dumps(secret_hits[:10])
        },
        {
            "gate":"No submission placeholders",
            "passed":
                len(placeholder_hits)==0,
            "detail":
                json.dumps(placeholder_hits[:20])
        }
    ])

else:
    audit_rows.append({
        "gate":"Repository package available",
        "passed":False,
        "detail":
            "Upload/extract the GitHub-ready repository ZIP to run release audit."
    })

release_audit_df=pd.DataFrame(audit_rows)
display(release_audit_df)

# Part XI — Final FCS readiness decision

The notebook separates:

- **scientific evidence readiness**
- **public release readiness**
- **human-validation status**

This prevents an optional follow-up from blocking a defensible short software note.

In [ ]:
# Scientific minimum:
core_science_ready=(
    not v3.empty and
    not v6.empty and
    not pub81.empty and
    not fixture_flags["v3_benchmark"] and
    not fixture_flags["v6_metrics"] and
    not fixture_flags["v81_pub"]
)

human_gate=(
    V7_DOUBLE_REVIEW_DONE
    if REQUIRE_V7_HUMAN_FOR_FCS
    else True
)

SCIENCE_READY_FOR_FCS=bool(
    core_science_ready and
    human_gate
)

# Release audit may be unavailable in a clean runtime.
if len(release_audit_df):
    release_known=not (
        len(release_audit_df)==1 and
        release_audit_df.iloc[0]["gate"]==
        "Repository package available"
    )

    if release_known:
        RELEASE_READY=bool(
            release_audit_df.passed.all()
        )
    else:
        RELEASE_READY=False
else:
    RELEASE_READY=False

status_df=pd.DataFrame([
    {
        "dimension":
            "Core scientific evidence",
        "ready":SCIENCE_READY_FOR_FCS,
        "detail":
            "v3 + v6 + corrected live v8.1 evidence"
    },
    {
        "dimension":
            "v8.2 IMERG multi-region empirical extension",
        "ready":V82_FINAL_READY,
        "detail":
            "Optional: include claim only when true"
    },
    {
        "dimension":
            "Independent v7 double-human review",
        "ready":V7_DOUBLE_REVIEW_DONE,
        "detail":
            "Recommended; only hard gate if REQUIRE_V7_HUMAN_FOR_FCS=True"
    },
    {
        "dimension":
            "Public GitHub release",
        "ready":RELEASE_READY,
        "detail":
            "Requires explicit license and removal of submission placeholders"
    }
])

display(status_df)

print(
    "SCIENCE_READY_FOR_FCS =",
    SCIENCE_READY_FOR_FCS
)

print(
    "RELEASE_READY =",
    RELEASE_READY
)

# Part XII — Generate manuscript-safe result text

This block automatically excludes unsupported claims.

In [ ]:
claim_lines=[]

if not v3.empty:
    claim_lines.append(
        "A frozen SciGeoGuard-X verifier was evaluated on "
        f"{len(v3)} controlled SilentGeoBench workflows. "
        "These generated reference labels test consistency with encoded scientific constraints "
        "and are not independent human ground truth."
    )

if not v6.empty:
    for _,r in v6.iterrows():
        sysname=str(r.get("system","LLM baseline"))
        acc=pd.to_numeric(
            r.get(
                "accuracy_vs_generated_reference",
                np.nan
            ),
            errors="coerce"
        )
        fscr=pd.to_numeric(
            r.get(
                "FSCR_vs_generated_reference",
                np.nan
            ),
            errors="coerce"
        )

        claim_lines.append(
            f"{sysname} achieved {acc:.2%} accuracy against the generated reference "
            f"and an FSCR of {fscr:.2%}."
        )

if not pub81.empty:
    for _,r in pub81.iterrows():
        exp=str(r.get("experiment",""))

        if "IMERG" in exp.upper():
            continue

        claim_lines.append(
            f"{exp}: primary effect={r.get('primary_effect',np.nan)}, "
            f"secondary effect={r.get('secondary_effect',np.nan)}."
        )

if V82_FINAL_READY:
    claim_lines.append(
        "Live GPM IMERG V07 recovery identified one rainy, temporally complete day in each of "
        "the four study regions; omitting the 0.5 h duration reproduced the expected 2.0× "
        "numeric overstatement, while labeling the full-day mean rate as accumulation gave 1/24 "
        "of the correct daily accumulation on rainy cells."
    )
else:
    claim_lines.append(
        "Multi-region IMERG recovery was not used as a main empirical claim because the final "
        "four-region rainy-day gate was not complete."
    )

if V7_DOUBLE_REVIEW_DONE:
    claim_lines.append(
        "Independent double review was completed; human agreement statistics are reported separately."
    )
else:
    claim_lines.append(
        "Independent blinded human review remains a separate pending validation layer and is not "
        "used to claim human-grounded accuracy."
    )

claims_text="\n\n".join(claim_lines)

print(claims_text)

(OUTPUTS/"FINAL_FCS_SAFE_CLAIMS.txt").write_text(
    claims_text,
    encoding="utf-8"
)

# Part XIII — Final export

Exports:

- `FINAL_EVIDENCE_TABLE.csv`
- `FINAL_REMAINING_WORK.csv`
- `FINAL_FCS_READINESS.csv`
- `FINAL_GITHUB_RELEASE_AUDIT.csv`
- `FINAL_FCS_SAFE_CLAIMS.txt`
- `FINAL_FCS_COMPOSITE_FIGURE_v9.png`
- optional v8.2 recovery results/log
- optional v7 human-agreement summary
- `FINAL_FILE_HASHES_SHA256.csv`
- one complete ZIP

In [ ]:
final_evidence_df.to_csv(
    OUTPUTS/"FINAL_EVIDENCE_TABLE.csv",
    index=False
)

remaining_df.to_csv(
    OUTPUTS/"FINAL_REMAINING_WORK.csv",
    index=False
)

status_df.to_csv(
    OUTPUTS/"FINAL_FCS_READINESS.csv",
    index=False
)

release_audit_df.to_csv(
    OUTPUTS/"FINAL_GITHUB_RELEASE_AUDIT.csv",
    index=False
)

evidence_check_df.to_csv(
    OUTPUTS/"FINAL_EVIDENCE_SOURCE_CHECKS.csv",
    index=False
)

if not human_summary.empty:
    human_summary.to_csv(
        OUTPUTS/"FINAL_V7_HUMAN_AGREEMENT.csv",
        index=False
    )

if 'v82_final' in globals() and not v82_final.empty:
    v82_final.to_csv(
        OUTPUTS/"FINAL_V8_2_IMERG_RESULTS.csv",
        index=False
    )

if 'imerg_log_df' in globals() and not imerg_log_df.empty:
    imerg_log_df.to_csv(
        OUTPUTS/"FINAL_V8_2_IMERG_SCOUT_LOG.csv",
        index=False
    )

final_status={
    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "science_ready_for_fcs":
        SCIENCE_READY_FOR_FCS,

    "v8_2_imerg_ready":
        V82_FINAL_READY,

    "v7_double_human_review_complete":
        V7_DOUBLE_REVIEW_DONE,

    "release_ready":
        RELEASE_READY,

    "require_v7_human_for_fcs":
        REQUIRE_V7_HUMAN_FOR_FCS,

    "license_choice":
        LICENSE_CHOICE,

    "recommended_submission_policy":
        (
            "Submit the Code & Data note using v3/v6/v8.1 evidence; include v8.2 only if its final gate passes. "
            "Do not claim human-grounded accuracy unless v7 is completed."
        )
}

(OUTPUTS/"FINAL_STATUS.json").write_text(
    json.dumps(
        final_status,
        indent=2
    ),
    encoding="utf-8"
)

# Hash outputs except the hash table itself.
hash_rows=[]

for p in sorted(OUTPUTS.rglob("*")):
    if (
        p.is_file() and
        p.name!="FINAL_FILE_HASHES_SHA256.csv"
    ):
        hash_rows.append({
            "file":
                str(p.relative_to(OUTPUTS)),
            "bytes":
                p.stat().st_size,
            "sha256":
                hashlib.sha256(
                    p.read_bytes()
                ).hexdigest()
        })

hash_df=pd.DataFrame(hash_rows)

hash_df.to_csv(
    OUTPUTS/"FINAL_FILE_HASHES_SHA256.csv",
    index=False
)

zip_path=shutil.make_archive(
    str(
        ROOT/
        "SciGeoGuard_X_v9_Final_Publication_Closure"
    ),
    "zip",
    root_dir=OUTPUTS
)

print("Created:",zip_path)
print(json.dumps(final_status,indent=2))

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

# Final interpretation

After v9, **do not create another experiment notebook by default**.

The only remaining actions should be administrative or independent validation:

- choose the software license;
- publish the cleaned GitHub repository;
- archive a release (e.g. DOI) if desired;
- replace manuscript placeholders;
- submit under manuscript type **Code & Data in Earth Science**;
- complete v7 human review later if you want a stronger independent-validation extension.

A new model experiment is not required for the current short FCS Code & Data note.